# SFT smoke quickstart

Clean entrypoint for the PLUM-style SFT stage. This notebook builds a tiny supervised next-item SID dataset, checks the key no-leakage and label-masking invariants, and contains a guarded smoke training cell.

The notebook is intentionally saved without outputs. Read it first, then run cells selectively.

## Imports and paths

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

import numpy as np
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

from src.sft import (
    SFTDataPaths,
    SFTDatasetBuilder,
    SFTExampleConfig,
    SFTSchema,
    SFTTrainingConfig,
    SIDMapping,
    evaluate_rankings,
    examples_to_dataset,
    generate_recommendations,
    train_sft,
)

ROOT

WindowsPath('C:/Users/User/plum-ml1m-repro')

In [2]:
SID_PATH = ROOT / "data" / "processed" / "SIDs_V1.npy"
CPT_ARTIFACT_DIR = ROOT / "data" / "processed" / "artifacts" / "CPT_user_behavior_V2"

# Tiny model is for pipeline smoke only, not for final SFT quality.
MODEL_PATH = CPT_ARTIFACT_DIR / "cpt_tiny_gpt2_smoke_50steps"
TOKENIZER_PATH = MODEL_PATH

# For a stronger real SFT smoke, switch MODEL_PATH/TOKENIZER_PATH to a GPT-2 CPT checkpoint.
# Example candidate:
# MODEL_PATH = ROOT / "data" / "processed" / "artifacts" / "CPT_user_behavior_V1" / "cpt_gpt2_v1_plus_plus" / "checkpoint-1000"
# TOKENIZER_PATH = MODEL_PATH

OUTPUT_DIR = ROOT / "data" / "processed" / "artifacts" / "SFT" / "sft_tiny_gpt2_smoke_100steps"

SID_PATH, MODEL_PATH, OUTPUT_DIR

(WindowsPath('C:/Users/User/plum-ml1m-repro/data/processed/SIDs_V1.npy'),
 WindowsPath('C:/Users/User/plum-ml1m-repro/data/processed/artifacts/CPT_user_behavior_V2/cpt_tiny_gpt2_smoke_50steps'),
 WindowsPath('C:/Users/User/plum-ml1m-repro/data/processed/artifacts/SFT/sft_tiny_gpt2_smoke_100steps'))

## Load tokenizer, SIDs, and build tiny SFT examples

In [3]:
sids = np.load(SID_PATH)
tokenizer = GPT2TokenizerFast.from_pretrained(TOKENIZER_PATH)

schema = SFTSchema(n_sid_levels=sids.shape[1])
builder = SFTDatasetBuilder(
    paths=SFTDataPaths(
        processed_dir=ROOT / "data" / "processed",
        raw_ml1m_dir=ROOT / "data" / "raw" / "ml-1m",
    ),
    schema=schema,
    config=SFTExampleConfig(
        max_history_events=50,
        min_history_events=3,
        max_length=512,
        include_user_features=True,
        include_ratings=True,
        train_examples_per_user=1,
        max_users=128,
    ),
)

train, val, test, users = builder.load_tables()
train_examples = builder.build_train_examples(sids=sids, tokenizer=tokenizer, train=train, users=users)
val_examples = builder.build_eval_examples(
    split="val",
    sids=sids,
    tokenizer=tokenizer,
    train=train,
    val=val,
    test=test,
    users=users,
)

len(train_examples), len(val_examples), train_examples[0].keys()

(128,
 128,
 dict_keys(['input_ids', 'labels', 'user_id', 'user_idx', 'split', 'history_item_idx', 'target_item_idx', 'target_sid', 'prompt_length']))

## Sanity checks: leakage and target-only labels

In [4]:
def assert_sft_example(example):
    assert example["target_item_idx"] not in example["history_item_idx"]
    assert len(example["target_sid"]) == sids.shape[1]
    assert len(example["input_ids"]) == len(example["labels"])
    assert any(label != -100 for label in example["labels"])
    first_target = next(i for i, label in enumerate(example["labels"]) if label != -100)
    assert first_target == example["prompt_length"]
    assert all(label == -100 for label in example["labels"][:first_target])

for example in train_examples[:32] + val_examples[:32]:
    assert_sft_example(example)

"sanity checks passed"

'sanity checks passed'

In [5]:
train_ds = examples_to_dataset(train_examples)
val_ds = examples_to_dataset(val_examples)
sid_mapping = SIDMapping.from_sids(sids, interactions=train)

{
    "train_examples": len(train_ds),
    "val_examples": len(val_ds),
    "sid_uniqueness": sid_mapping.uniqueness,
    "collision_buckets": sid_mapping.n_collision_buckets,
    "collided_items_in_collision_buckets": sid_mapping.n_collided_items,
    "collision_excess": sid_mapping.n_collision_excess,
}

{'train_examples': 128,
 'val_examples': 128,
 'sid_uniqueness': 0.9495412844036697,
 'collision_buckets': 98,
 'collided_items_in_collision_buckets': 285,
 'collision_excess': 187}

## Guarded SFT smoke training

Set `RUN_TRAINING = True` when you want to launch the smoke run. The default is `False` so an accidental Run All does not start fine-tuning.

In [6]:
RUN_TRAINING = False

training_config = SFTTrainingConfig(
    model_name_or_path=MODEL_PATH,
    tokenizer_name_or_path=TOKENIZER_PATH,
    output_dir=OUTPUT_DIR,
    max_steps=100,
    learning_rate=1e-5,
    warmup_ratio=0.0,
    weight_decay=0.0,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    eval_steps=25,
    save_steps=25,
    logging_steps=10,
    fp16=False,
    bf16=False,
    gradient_checkpointing=False,
    save_total_limit=1,
)

if RUN_TRAINING:
    result = train_sft(train_ds, val_ds, training_config)
    result["train_output"].metrics
else:
    print("Set RUN_TRAINING = True to launch SFT smoke training.")

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
25,10.804391,10.803997
50,10.802991,10.803719
75,10.804597,10.803532
100,10.804248,10.803463


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

mtime may not be reliable on this filesystem, falling back to numerical ordering


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Optional decoding and ranking check

Run this after a smoke checkpoint exists. It uses trie-constrained decoding and computes small validation metrics on a tiny subset.

In [7]:
RUN_EVAL = False

if RUN_EVAL:
    model_dir = OUTPUT_DIR if (OUTPUT_DIR / "model.safetensors").exists() else MODEL_PATH
    eval_tokenizer = GPT2TokenizerFast.from_pretrained(model_dir)
    model = GPT2LMHeadModel.from_pretrained(model_dir)
    model.eval()

    records = []
    for example in val_examples[:32]:
        record = generate_recommendations(
            model=model,
            tokenizer=eval_tokenizer,
            example=example,
            sid_mapping=sid_mapping,
            schema=schema,
            sids=sids,
            k=20,
            num_beams=20,
            constraint="trie",
        )
        records.append(record)

    metrics = evaluate_rankings(records)
    metrics
else:
    print("Set RUN_EVAL = True after training to run decoding metrics.")

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]